<a href="https://colab.research.google.com/github/kashaf11303/urdu-ocr-codesaviours-si26-kashaf/blob/main/SI26_Week3_kashaf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python-headless pillow
import cv2
import numpy as np
from PIL import Image
import os
import matplotlib.pyplot as plt
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
import cv2
import os

def preprocess_image(image_path, save_path):
    img = cv2.imread(image_path)

    if img is None:
        print(f'Could not load: {image_path}')
        return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (512, 128))
    denoised = cv2.fastNlMeansDenoising(resized, h=10)
    _, binary = cv2.threshold(
        denoised,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    cv2.imwrite(save_path, binary)

    return binary

In [ ]:
import os

input_folder = "/content/drive/MyDrive/200 images"
output_folder = "/content/drive/MyDrive/processed_images"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.join(input_folder, filename)
        save_path = os.path.join(output_folder, filename)

        preprocess_image(image_path, save_path)

print("✅ All images have been preprocessed successfully!")

✅ All images have been preprocessed successfully!


In [ ]:
!pip install transformers torch pillow pandas

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

In [ ]:
!pip install sentencepiece tiktoken

In [ ]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten"
)

print("Processor loaded successfully!")

Processor loaded successfully!


In [ ]:
print(UrduOCRDataset)

<class '__main__.UrduOCRDataset'>


In [ ]:
import pandas as pd
import os

df = pd.read_csv(csv_path)

df["image"] = df["image"].apply(
    lambda x: "/content/drive/MyDrive/200 images/" + os.path.basename(x)
)

df.to_csv("/content/drive/MyDrive/labels_fixed.csv", index=False)

print("labels_fixed.csv created successfully")

labels_fixed.csv created successfully


In [ ]:
csv_path = "/content/drive/MyDrive/labels_fixed.csv"

In [ ]:
!pip install transformers torch pillow pandas

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd


class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        # Load and convert image
        image = Image.open(row['image']).convert("RGB")

        # Process image for the model
        encoding = self.processor(
            image,
            return_tensors="pt"
        )
        pixel_values = encoding.pixel_values.squeeze()

        # Process the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding="max_length",
            max_length=128
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [ ]:
# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-handwritten",
    use_fast=False
)

# Create dataset
dataset = UrduOCRDataset(csv_path, processor)

# Test it loads correctly
sample = dataset[0]

print("Sample pixel_values shape:", sample["pixel_values"].shape)
print("Sample labels shape:", sample["labels"].shape)
print("Dataset is working correctly!")

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

print(f"Training samples: {train_size}")
print(f"Testing samples: {test_size}")

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40
